In [1]:
import numpy as np
from pathlib import Path

def find_project_root(marker="requirements.txt"):
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError("root not found")

GOES_DIR = find_project_root() / "data" / "raw" / "goes"
files = sorted(GOES_DIR.glob("goes_*.npz"))
print(f"Day-files on disk: {len(files)}")
if files:
    print(f"Date range: {files[0].stem.replace('goes_','')} -> {files[-1].stem.replace('goes_','')}")
    # peek inside one
    d = np.load(files[0])
    print(f"Entries in first file: {len(d.keys())}  (expect 48: 24h x 2 cities)")
    k = list(d.keys())[0]
    print(f"Sample key: {k}   shape: {d[k].shape}")
    # total patches available
    total = sum(len(np.load(f).keys()) for f in files)
    print(f"Total patches available right now: {total}")

Day-files on disk: 224
Date range: 2025-09-19 -> 2026-04-30
Entries in first file: 48  (expect 48: 24h x 2 cities)
Sample key: kingston|2025-09-19T00:00:00   shape: (3, 64, 64)
Total patches available right now: 10752


In [2]:
import sys

def find_project_root(marker="requirements.txt"):
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if (p / marker).exists(): return p
    raise RuntimeError("root not found")

sys.path.insert(0, str(find_project_root()))

from src.models.goes_dataset import GoesPatchDataset, IR_ONLY, ALL_BANDS

# Build the test split (the only one with imagery on disk right now)
ds = GoesPatchDataset(split="test", channels=IR_ONLY)
print(f"Usable test examples: {len(ds)}")
print(f"Normalization stats (train-derived in real runs): {ds.norm_stats}")

x, y = ds[0]
print(f"\nOne sample -> image {tuple(x.shape)}, label {y.item()}")
print(f"Image mean ~0 after norm: {x.mean().item():.3f}, std ~1: {x.std().item():.3f}")

# Label balance in the usable subset
labels = [ds.label_of[k] for k in ds.keys]
print(f"\nPositive rate in usable test patches: {sum(labels)/len(labels):.3f}")
print(f"(expect ~0.10, matching the tabular test split)")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\romay\Projects\Python-Projects\smart-patio-shield\.venv\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.